In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

In [3]:
np.random.seed(42)

normal_data = np.random.normal(50, 15, 800)

skewed_data = np.random.exponential(2, 800) * 10 + 20

outliers = [200, 180, 190, 210, 195]

data = np.concatenate([normal_data, skewed_data, outliers])

df = pd.DataFrame({"original": data})

In [5]:
df.describe()

,original
count,1605.000000
mean,45.650689
std,20.520205
min,1.380990
25%,30.673327
50%,42.812386
75%,55.987042
max,210.000000


In [6]:
scalers = {
    'MinMax': MinMaxScaler(),
    'Standard': StandardScaler(),
    'Robust': RobustScaler()
}
 
for name, scaler in scalers.items():
    df[name] = scaler.fit_transform(df[['original']]).flatten()
 
# Check what we're working with
print("Original Data Stats:")
print(f"Mean: {df['original'].mean():.2f}")
print(f"Median: {df['original'].median():.2f}")
print(f"Std Dev: {df['original'].std():.2f}")
print(f"Skewness: {stats.skew(df['original']):.2f}")
print(f"Range: {df['original'].min():.1f} to {df['original'].max():.1f}")

Original Data Stats:
Mean: 45.65
Median: 42.81
Std Dev: 20.52
Skewness: 2.07
Range: 1.4 to 210.0


#### MinMaxScaler Analysis

In [7]:
min_ = df['original'].min()
max_ = df['original'].max()
print(f"Scaling range: {min_:.1f} to {max_:.1f}")
 
# Show the compression effect
percentiles = [50, 75, 90, 95, 99]
for p in percentiles:
    pct_val = df['MinMax'].quantile(p/100)
    print(f"{p}% of data falls below: {pct_val:.3f}")
 
data_below_half = (df['MinMax'] < 0.5).sum() / len(df) * 100
print(f"\nResult: {data_below_half:.1f}% of data compressed below 0.5")

Scaling range: 1.4 to 210.0
50% of data falls below: 0.199
75% of data falls below: 0.262
90% of data falls below: 0.319
95% of data falls below: 0.368
99% of data falls below: 0.541

Result: 98.6% of data compressed below 0.5


In [10]:
df['original'].describe()

count    1605.000000
mean       45.650689
std        20.520205
min         1.380990
25%        30.673327
50%        42.812386
75%        55.987042
max       210.000000
Name: original, dtype: float64

#### StandardScaler Analysis

In [11]:
mean_orig = df['original'].mean()
std_orig = df['original'].std()
 
# Compare with/without outliers
clean_data = df['original'][df['original'] < 150]
mean_clean = clean_data.mean()
std_clean = clean_data.std()
 
print(f"With outliers: mean={mean_orig:.2f}, std={std_orig:.2f}")
print(f"Without outliers: mean={mean_clean:.2f}, std={std_clean:.2f}")
print(f"Outlier impact: mean +{mean_orig - mean_clean:.2f}, std +{std_orig - std_clean:.2f}")
 
# Show impact on typical data points
typical_value = 50
z_with_outliers = (typical_value - mean_orig) / std_orig
z_without_outliers = (typical_value - mean_clean) / std_clean
print(f"\nZ-score for value 50:")
print(f"With outliers: {z_with_outliers:.2f}")
print(f"Without outliers: {z_without_outliers:.2f}")

With outliers: mean=45.65, std=20.52
Without outliers: mean=45.11, std=18.51
Outlier impact: mean +0.54, std +2.01

Z-score for value 50:
With outliers: 0.21
Without outliers: 0.26


#### RobustScaler Analysis

In [12]:
median_orig = df['original'].median()
q25, q75 = df['original'].quantile([0.25, 0.75])
iqr = q75 - q25
 
# Compare with/without outliers
clean_data = df['original'][df['original'] < 150]
median_clean = clean_data.median()
q25_clean, q75_clean = clean_data.quantile([0.25, 0.75])
iqr_clean = q75_clean - q25_clean
 
print(f"With outliers: median={median_orig:.2f}, IQR={iqr:.2f}")
print(f"Without outliers: median={median_clean:.2f}, IQR={iqr_clean:.2f}")
print(f"Outlier impact: median {abs(median_orig - median_clean):.2f}, IQR {abs(iqr - iqr_clean):.2f}")
 
# Show consistency for typical data points
typical_value = 50
robust_with_outliers = (typical_value - median_orig) / iqr
robust_without_outliers = (typical_value - median_clean) / iqr_clean
print(f"\nRobust score for value 50:")
print(f"With outliers: {robust_with_outliers:.2f}")
print(f"Without outliers: {robust_without_outliers:.2f}")

With outliers: median=42.81, IQR=25.31
Without outliers: median=42.80, IQR=25.08
Outlier impact: median 0.01, IQR 0.24

Robust score for value 50:
With outliers: 0.28
Without outliers: 0.29
